<a href="https://colab.research.google.com/github/Adamphoenix003/GNN-LinkPrediction/blob/main/SEAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.0 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import GCNConv
from torch_geometric.utils import k_hop_subgraph, to_networkx

import numpy as np
import networkx as nx
from tqdm import tqdm

In [3]:
dataset = Planetoid(root='data/Planetoid', name='Cora')
data = dataset[0]

# Create link prediction split
transform = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,
    split_labels=True
)

train_data, val_data, test_data = transform(data)

print(train_data)

Processing...


Data(x=[2708, 1433], edge_index=[2, 8448], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708], pos_edge_label=[4224], pos_edge_label_index=[2, 4224], neg_edge_label=[4224], neg_edge_label_index=[2, 4224])


Done!


In [4]:
def drnl_node_labeling(edge_index, src, dst, num_nodes):
    G = to_networkx(data, to_undirected=True)

    # Shortest path distances
    dist_to_src = nx.single_source_shortest_path_length(G, src)
    dist_to_dst = nx.single_source_shortest_path_length(G, dst)

    labels = []

    for i in range(num_nodes):
        d1 = dist_to_src.get(i, 1e6)
        d2 = dist_to_dst.get(i, 1e6)

        if d1 == 1e6 or d2 == 1e6:
            labels.append(0)
        else:
            labels.append(1 + min(d1, d2))

    return torch.tensor(labels)

In [5]:
def extract_subgraph(src, dst, edge_index, x, y, hops=2):

    subset, sub_edge_index, mapping, _ = k_hop_subgraph(
        [src, dst],
        hops,
        edge_index,
        relabel_nodes=True
    )

    sub_x = x[subset]

    # DRNL labels
    z = drnl_node_labeling(edge_index, src, dst, x.size(0))
    sub_z = z[subset]

    return sub_x, sub_edge_index, sub_z

In [6]:
def build_seal_dataset(split_data):

    pos_edge = split_data.pos_edge_label_index.t()
    neg_edge = split_data.neg_edge_label_index.t()

    graphs = []
    labels = []

    print("Extracting subgraphs...")

    for edge in tqdm(pos_edge):
        src, dst = edge.tolist()
        sub_x, sub_edge_index, sub_z = extract_subgraph(
            src, dst,
            train_data.edge_index,
            data.x,
            1
        )

        graphs.append((sub_x, sub_edge_index, sub_z))
        labels.append(1)

    for edge in tqdm(neg_edge):
        src, dst = edge.tolist()
        sub_x, sub_edge_index, sub_z = extract_subgraph(
            src, dst,
            train_data.edge_index,
            data.x,
            0
        )

        graphs.append((sub_x, sub_edge_index, sub_z))
        labels.append(0)

    return graphs, torch.tensor(labels)

In [7]:
train_graphs, train_labels = build_seal_dataset(train_data)

Extracting subgraphs...


100%|██████████| 4224/4224 [03:51<00:00, 18.26it/s]


In [8]:
class SEAL(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(SEAL, self).__init__()

        self.conv1 = GCNConv(in_channels + 1, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)

        self.lin1 = nn.Linear(hidden_channels, hidden_channels)
        self.lin2 = nn.Linear(hidden_channels, 1)

    def forward(self, x, edge_index, z):

        z = z.unsqueeze(1).float()
        x = torch.cat([x, z], dim=1)

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        # Global pooling (mean)
        x = torch.mean(x, dim=0)

        x = self.lin1(x)
        x = F.relu(x)

        x = self.lin2(x)

        return torch.sigmoid(x)

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = SEAL(dataset.num_features, 64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

def train():
    model.train()
    total_loss = 0

    for (sub_x, sub_edge_index, sub_z), label in zip(train_graphs, train_labels):

        sub_x = sub_x.to(device)
        sub_edge_index = sub_edge_index.to(device)
        sub_z = sub_z.to(device)
        label = torch.tensor([label], dtype=torch.float).to(device)

        optimizer.zero_grad()

        out = model(sub_x, sub_edge_index, sub_z)

        loss = criterion(out, label)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_graphs)

In [10]:
for epoch in range(1, 11):
    loss = train()
    print(f"Epoch {epoch}, Loss: {loss:.4f}")

Epoch 1, Loss: 1.0891
Epoch 2, Loss: 1.1571
Epoch 3, Loss: 0.1575
Epoch 4, Loss: 1.1511
Epoch 5, Loss: 1.1885
Epoch 6, Loss: 0.1582
Epoch 7, Loss: 0.4576
Epoch 8, Loss: 0.1581
Epoch 9, Loss: 0.1814
Epoch 10, Loss: 1.3620


In [11]:
from sklearn.metrics import roc_auc_score, average_precision_score

In [12]:
@torch.no_grad()
def evaluate(graphs, labels):

    model.eval()

    preds = []
    true_labels = []

    for (sub_x, sub_edge_index, sub_z), label in zip(graphs, labels):

        sub_x = sub_x.to(device)
        sub_edge_index = sub_edge_index.to(device)
        sub_z = sub_z.to(device)

        out = model(sub_x, sub_edge_index, sub_z)

        preds.append(out.item())
        true_labels.append(label.item())

    preds = np.array(preds)
    true_labels = np.array(true_labels)

    auc = roc_auc_score(true_labels, preds)
    ap = average_precision_score(true_labels, preds)

    return auc, ap

In [13]:
train_graphs, train_labels = build_seal_dataset(train_data)

Extracting subgraphs...


100%|██████████| 4224/4224 [04:04<00:00, 17.28it/s]


In [14]:
val_graphs, val_labels = build_seal_dataset(val_data)
test_graphs, test_labels = build_seal_dataset(test_data)

Extracting subgraphs...


100%|██████████| 527/527 [00:29<00:00, 17.82it/s]


Extracting subgraphs...


100%|██████████| 527/527 [00:29<00:00, 17.83it/s]


In [15]:
val_auc, val_ap = evaluate(val_graphs, val_labels)
test_auc, test_ap = evaluate(test_graphs, test_labels)

print("Validation AUC :", val_auc)
print("Validation AP  :", val_ap)

print("Test AUC       :", test_auc)
print("Test AP        :", test_ap)

Validation AUC : 0.5259767615193228
Validation AP  : 0.5209558501883527
Test AUC       : 0.513911042779112
Test AP        : 0.5017409838574504
